# Compare bots on the same dataset

This notebook reproduces what the Flutter **Compare** tab does, but from Python:

1. Generate a single candle dataset (synthetic 1h sine pattern).
2. Run every bot in `BOT_REGISTRY` against it with default-ish parameters, including a `DSLBot` defined inline.
3. Plot all equity curves on a shared axis and print a comparison table.

Pair this with `01_quickstart.ipynb` if you want to learn the SDK first.

In [ ]:
import math
from decimal import Decimal

import matplotlib.pyplot as plt
import pandas as pd

from backtester import (
    BacktestConfig,
    BacktestEngine,
    Candle,
    BollingerReversion,
    DorothyDCA,
    DSLBot,
    EMACross,
    MACDCross,
    RSIReversion,
)


## 1. Build a deterministic dataset

In [ ]:
def candles(n: int = 720) -> list[Candle]:
    base = 100.0
    out = []
    for i in range(n):
        offset = 25 * math.sin(i / 20.0) + i * 0.02
        close = base + offset
        open_ = base + 25 * math.sin((i - 1) / 20.0) + (i - 1) * 0.02 if i > 0 else close
        high = max(open_, close) + 0.5
        low = min(open_, close) - 0.5
        out.append(
            Candle(
                timestamp_ms=i * 3_600_000,
                open=Decimal(str(open_)),
                high=Decimal(str(high)),
                low=Decimal(str(low)),
                close=Decimal(str(close)),
                volume=Decimal("100"),
            )
        )
    return out

dataset = candles()
len(dataset), float(dataset[0].close), float(dataset[-1].close)

## 2. Run each bot

We instantiate each strategy with parameters that play well on a sine pattern and run them all through a fresh `BacktestEngine`.

In [ ]:
DSL_TEXT = """
name: ema + rsi filter
indicators:
  - ema(close, 12) as fast
  - ema(close, 26) as slow
  - rsi(close, 14) as rsi14
entry:
  long: fast > slow AND rsi14 < 70
exit:
  long: fast < slow OR rsi14 > 80
risk:
  stop_loss_pct: 0.05
  take_profit_pct: 0.10
  size_pct: 2.0
""".strip()

bot_setups = [
    ("EMACross",           EMACross(fast_ema=12, slow_ema=26, stop_loss_pct=0.05, profit_factor=0.05)),
    ("RSIReversion",       RSIReversion()),
    ("MACDCross",          MACDCross()),
    ("BollingerReversion", BollingerReversion()),
    ("DorothyDCA",         DorothyDCA()),
    ("DSL (ema+rsi)",      DSLBot(dsl_text=DSL_TEXT)),
]

cfg = BacktestConfig(
    initial_cash=Decimal("10000"),
    taker_fee_pct=Decimal("0.1"),
    slippage_pct=Decimal("0.05"),
)

runs = {}
for name, bot in bot_setups:
    engine = BacktestEngine(cfg)
    result = engine.run(bot, dataset, symbol="TESTUSDT", timeframe="1h", bot_names=[name])
    runs[name] = result
    s = result.summary()
    print(f"{name:24s}  return={s['total_return_pct']:7.2f}%  trades={s['trades']:3d}  DD={s['max_drawdown_pct']:5.2f}%")

## 3. Plot equity curves on a shared axis

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for name, result in runs.items():
    eq = [float(e) for e in result.equity_curve]
    ax.plot(eq, label=name, linewidth=1.5)
ax.axhline(10000, color="grey", linestyle="--", linewidth=0.8)
ax.set_xlabel("Candle index")
ax.set_ylabel("Equity (USDT)")
ax.set_title("Equity curves — same dataset, different bots")
ax.legend(loc="best")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Comparison table

In [ ]:
rows = []
for name, result in runs.items():
    s = result.summary()
    rows.append({
        "bot": name,
        "return_pct": round(s["total_return_pct"], 2),
        "max_dd_pct": round(s["max_drawdown_pct"], 2),
        "win_rate_pct": round(s["win_rate_pct"], 2),
        "profit_factor": round(s["profit_factor"], 3),
        "trades": s["trades"],
        "final_equity": round(s["final_equity"], 2),
    })
df = pd.DataFrame(rows).sort_values("return_pct", ascending=False).reset_index(drop=True)
df